# 🫀 Heart Disease Dataset — Exploratory Data Analysis

**Dataset:** Cleveland Heart Disease (UCI / Kaggle variant)  
**Rows:** 1025 | **Columns:** 14 (13 features + 1 target)  
**Goal:** Understand distributions, relationships, and class balance before modelling.

---
### Column Reference
| Column | Description |
|--------|-------------|
| `age` | Age in years |
| `sex` | 1 = male, 0 = female |
| `cp` | Chest pain type (0–3) |
| `trestbps` | Resting blood pressure (mm Hg) |
| `chol` | Serum cholesterol (mg/dl) |
| `fbs` | Fasting blood sugar > 120 mg/dl (1 = true) |
| `restecg` | Resting ECG results (0–2) |
| `thalach` | Maximum heart rate achieved |
| `exang` | Exercise-induced angina (1 = yes) |
| `oldpeak` | ST depression induced by exercise |
| `slope` | Slope of peak exercise ST segment (0–2) |
| `ca` | Number of major vessels coloured by fluoroscopy (0–3) |
| `thal` | Thalassemia type (0–3) |
| `target` | **1 = heart disease, 0 = no heart disease** |


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110

df = pd.read_csv("heart_disease.csv")
print(f"Shape: {df.shape}")
df.head()


## 1. Basic Info & Data Types

In [ ]:
df.info()


In [ ]:
df.describe().T.style.background_gradient(cmap="Blues", subset=["mean","std","min","max"])


## 2. Missing Values

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "✅  No missing values found.")


## 3. Target Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count plot
counts = df["target"].value_counts()
axes[0].bar(["No Disease (0)", "Disease (1)"], counts.values, color=["#4C72B0","#DD8452"], edgecolor="white", width=0.5)
axes[0].set_title("Target Class Counts")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

# Pie
axes[1].pie(counts.values, labels=["No Disease","Disease"], autopct="%1.1f%%",
            colors=["#4C72B0","#DD8452"], startangle=90, wedgeprops=dict(edgecolor="white"))
axes[1].set_title("Class Balance")

plt.suptitle("Heart Disease Target Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Continuous Feature Distributions

In [ ]:
continuous = ["age", "trestbps", "chol", "thalach", "oldpeak"]

fig, axes = plt.subplots(2, len(continuous), figsize=(18, 7))

for i, col in enumerate(continuous):
    # Histogram with KDE
    axes[0, i].hist(df[col], bins=25, color="#4C72B0", edgecolor="white", alpha=0.85)
    axes[0, i].set_title(col, fontsize=11)
    axes[0, i].set_ylabel("Count")

    # Boxplot by target
    groups = [df[df["target"]==0][col], df[df["target"]==1][col]]
    bp = axes[1, i].boxplot(groups, patch_artist=True,
                             labels=["No Disease","Disease"],
                             medianprops=dict(color="black", linewidth=2))
    for patch, color in zip(bp["boxes"], ["#4C72B0","#DD8452"]):
        patch.set_facecolor(color)
    axes[1, i].set_title(f"{col} by Target")

plt.suptitle("Continuous Feature Distributions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 5. Categorical Feature Analysis

In [ ]:
categorical = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(categorical):
    ct = df.groupby([col, "target"]).size().unstack(fill_value=0)
    ct.plot(kind="bar", ax=axes[i], color=["#4C72B0","#DD8452"],
            edgecolor="white", rot=0, legend=(i == 0))
    axes[i].set_title(col)
    axes[i].set_xlabel("")
    if i == 0:
        axes[i].legend(["No Disease","Disease"], fontsize=8)

plt.suptitle("Categorical Features vs Target", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Top correlations with target
print("\nTop correlations with target:")
print(corr["target"].drop("target").abs().sort_values(ascending=False))


## 7. Pairplot (Continuous Features)

In [ ]:
pp_df = df[continuous + ["target"]].copy()
pp_df["target"] = pp_df["target"].map({0:"No Disease", 1:"Disease"})
g = sns.pairplot(pp_df, hue="target", palette={"No Disease":"#4C72B0","Disease":"#DD8452"},
                 diag_kind="kde", plot_kws={"alpha": 0.5, "s": 20})
g.fig.suptitle("Pairplot of Continuous Features", y=1.02, fontsize=13, fontweight="bold")
plt.show()


## 8. Statistical Tests (Mann-Whitney U)

In [ ]:
print(f"{'Feature':<12} {'Mean(No)':>10} {'Mean(Yes)':>10} {'p-value':>12} {'Significant':>12}")
print("-" * 58)
for col in continuous:
    g0 = df[df["target"]==0][col]
    g1 = df[df["target"]==1][col]
    stat, p = stats.mannwhitneyu(g0, g1, alternative="two-sided")
    sig = "✅ Yes" if p < 0.05 else "❌ No"
    print(f"{col:<12} {g0.mean():>10.2f} {g1.mean():>10.2f} {p:>12.4f} {sig:>12}")


## 9. EDA Summary

| Insight | Detail |
|---------|--------|
| **Class balance** | Near-balanced: 51.3% disease, 48.7% no disease |
| **Strongest positive correlates** | `cp` (chest pain type), `thalach` (max HR), `slope` |
| **Strongest negative correlates** | `exang`, `oldpeak`, `ca`, `thal` |
| **Age** | Patients with disease are slightly younger (mean ~52 vs ~57) |
| **Sex** | Males are over-represented; females more likely to have disease in this sample |
| **No missing values** | Dataset is clean — no imputation needed |

➡️ Proceed to `02_preprocessing.ipynb` for feature engineering and scaling.
